In [1]:
import importlib, env
importlib.reload(env)
from env import CloudClusterEnv, STEPS_PER_WEEK

import json
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal

stats = json.load(open('trace_params.json'))['stats']
print("Setup ready.")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Setup complete. Steps per week: 672
CloudClusterEnv defined.
Setup ready.


In [2]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim=32, action_dim=1):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(),
            nn.Linear(256, 256),       nn.Tanh(),
        )
        self.actor_mean = nn.Linear(256, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        self.critic = nn.Linear(256, 1)

    def forward(self, state):
        x = self.shared(state)
        return self.actor_mean(x), self.critic(x)

    def get_action(self, state):
        mean, value = self.forward(state)
        std = torch.exp(self.log_std)
        dist = Normal(mean, std)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        return action, log_prob, value.squeeze(-1)

    def evaluate_actions(self, state, action):
        mean, value = self.forward(state)
        std = torch.exp(self.log_std)
        dist = Normal(mean, std)
        log_prob = dist.log_prob(action).sum(-1)
        entropy = dist.entropy().sum(-1)
        return log_prob, value.squeeze(-1), entropy

print("ActorCritic defined.")

ActorCritic defined.


In [3]:
def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0.0
    values = values + [next_value]
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t+1] * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
    returns = [a + v for a, v in zip(advantages, values[:-1])]
    return advantages, returns

def ppo_update(net, optimizer, states, actions, old_log_probs, advantages, returns,
               clip_eps=0.2, epochs=4, value_coef=0.5, entropy_coef=0.01):
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    for _ in range(epochs):
        new_log_probs, values, entropy = net.evaluate_actions(states, actions)
        ratio = torch.exp(new_log_probs - old_log_probs)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()
        critic_loss = ((values - returns)**2).mean()
        loss = actor_loss + value_coef*critic_loss - entropy_coef*entropy.mean()
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 0.5)
        optimizer.step()
    return actor_loss.item(), critic_loss.item(), entropy.mean().item()

def train(env_instance, net, optimizer, total_steps=250_000, rollout_len=2048):
    state, _ = env_instance.reset()
    state = torch.tensor(state, dtype=torch.float32)
    steps_done = 0
    episode_reward = 0
    episode_rewards = []
    while steps_done < total_steps:
        states, actions, log_probs = [], [], []
        rewards, values, dones = [], [], []
        for _ in range(rollout_len):
            with torch.no_grad():
                action, log_prob, value = net.get_action(state.unsqueeze(0))
            a = action.squeeze(0).numpy()
            next_state, reward, done, tr, info = env_instance.step(a)
            states.append(state); actions.append(action.squeeze(0))
            log_probs.append(log_prob.squeeze(0)); rewards.append(float(reward))
            values.append(value.item()); dones.append(1.0 if done else 0.0)
            episode_reward += reward
            state = torch.tensor(next_state, dtype=torch.float32)
            steps_done += 1
            if done:
                episode_rewards.append(episode_reward); episode_reward = 0
                state, _ = env_instance.reset()
                state = torch.tensor(state, dtype=torch.float32)
        with torch.no_grad():
            _, _, next_value = net.get_action(state.unsqueeze(0))
            next_value = next_value.item()
        advantages, returns = compute_gae(rewards, values, dones, next_value)
        b_states = torch.stack(states); b_actions = torch.stack(actions)
        b_old_lp = torch.stack(log_probs)
        b_adv = torch.tensor(advantages, dtype=torch.float32)
        b_ret = torch.tensor(returns, dtype=torch.float32)
        ppo_update(net, optimizer, b_states, b_actions, b_old_lp, b_adv, b_ret)
        recent = np.mean(episode_rewards[-5:]) if episode_rewards else float('nan')
        print(f"steps {steps_done:6d} | recent ep reward {recent:8.1f}")
    return episode_rewards

print("Training functions defined.")

Training functions defined.


In [4]:
def eval_on_surges(net, n_episodes=10, enable_hints=False, force_zero_hints=False, seed_base=1000):
    breaches_list = []
    for i in range(n_episodes):
        e = CloudClusterEnv(stats, enable_surges=True, enable_hints=enable_hints, seed=seed_base+i)
        obs, _ = e.reset()
        obs = torch.tensor(obs, dtype=torch.float32)
        ep_breach = 0
        for t in range(STEPS_PER_WEEK):
            if force_zero_hints:
                obs[-3:] = 0.0   # blank the hint slots
            with torch.no_grad():
                mean, _ = net.forward(obs.unsqueeze(0))
            obs, r, done, tr, info = e.step(mean.squeeze(0).numpy())
            obs = torch.tensor(obs, dtype=torch.float32)
            ep_breach += info['breaches']
            if done: break
        breaches_list.append(ep_breach)
    return np.mean(breaches_list), np.std(breaches_list)

# load the existing hint-naive agent
net_naive = ActorCritic()
net_naive.load_state_dict(torch.load('ppo_sla-focused.pth'))
net_naive.eval()

mean_b, std_b = eval_on_surges(net_naive, n_episodes=10, enable_hints=False)
print(f"BASELINE — hint-naive agent on surge environment:")
print(f"  Mean breaches/week: {mean_b:.0f} (std {std_b:.0f})")

BASELINE — hint-naive agent on surge environment:
  Mean breaches/week: 6713 (std 10726)


In [5]:
# train a fresh agent WITH surges and hints enabled
env_hint = CloudClusterEnv(stats, enable_surges=True, enable_hints=True, min_surges=8, max_surges=12, seed=None)
net_hint = ActorCritic()
optimizer = torch.optim.Adam(net_hint.parameters(), lr=3e-4)

print("Training hint-aware agent (250k steps, surges + hints on)...\n")
episode_rewards = train(env_hint, net_hint, optimizer, total_steps=250_000)

torch.save(net_hint.state_dict(), 'ppo_hint_aware.pth')
print("\nSaved ppo_hint_aware.pth")
print(f"First episode reward: {episode_rewards[0]:.1f}")
print(f"Last episode reward:  {episode_rewards[-1]:.1f}")

Training hint-aware agent (250k steps, surges + hints on)...

steps   2048 | recent ep reward   -223.5
steps   4096 | recent ep reward   -217.8
steps   6144 | recent ep reward   -199.7
steps   8192 | recent ep reward   -193.6
steps  10240 | recent ep reward   -190.7
steps  12288 | recent ep reward   -193.2
steps  14336 | recent ep reward   -201.2
steps  16384 | recent ep reward   -215.2
steps  18432 | recent ep reward   -197.4
steps  20480 | recent ep reward   -205.1
steps  22528 | recent ep reward   -217.3
steps  24576 | recent ep reward   -216.8
steps  26624 | recent ep reward   -213.3
steps  28672 | recent ep reward   -214.9
steps  30720 | recent ep reward   -219.7
steps  32768 | recent ep reward   -223.7
steps  34816 | recent ep reward   -206.6
steps  36864 | recent ep reward   -198.9
steps  38912 | recent ep reward   -199.5
steps  40960 | recent ep reward   -198.8
steps  43008 | recent ep reward   -196.0
steps  45056 | recent ep reward   -199.2
steps  47104 | recent ep reward   -1

In [6]:
# load the hint-aware agent
net_hint = ActorCritic()
net_hint.load_state_dict(torch.load('ppo_hint_aware.pth'))
net_hint.eval()

N = 30   # more episodes to average out the high surge-placement variance

# same seeds for both conditions → identical surges → fair comparison
with_hints, _    = eval_on_surges(net_hint, n_episodes=N, enable_hints=True,
                                   force_zero_hints=False, seed_base=2000)
without_hints, _ = eval_on_surges(net_hint, n_episodes=N, enable_hints=True,
                                   force_zero_hints=True,  seed_base=2000)

print("="*50)
print("HINT EXPERIMENT — same agent, same surges (30 weeks)")
print("="*50)
print(f"  WITH hints active:     {with_hints:>8.0f} breaches/week")
print(f"  WITHOUT hints (blanked): {without_hints:>8.0f} breaches/week")
print("="*50)
if without_hints > 0:
    reduction = (without_hints - with_hints) / without_hints * 100
    print(f"\n  Hints reduce surge breaches by {reduction:.1f}%")

HINT EXPERIMENT — same agent, same surges (30 weeks)
  WITH hints active:         8214 breaches/week
  WITHOUT hints (blanked):     8483 breaches/week

  Hints reduce surge breaches by 3.2%
